# Milestone 1 - the state a graph passes between its nodes

**Why this comes first.** Every node in a LangGraph app is a function that takes
the state and returns a *partial* update - just the keys it changed. LangGraph
merges that update into the state for you. Once that clicks, the rest of the
framework is bookkeeping.

**What to do.** Fill in the two TypedDicts, then implement `merge_update`, which
is a simplified version of what LangGraph does internally on every node return,
and `bump_attempts`, a node-shaped function that returns a partial update.

**New to coding?** Do `m0_python_basics.ipynb` first. Words you'll meet here: TypedDict, type hint, mutate, partial update, state. Each one is explained in plain language in [docs/CONCEPTS.md](../docs/CONCEPTS.md), along with Big-O and a routine for getting started when the cell is blank.

**How to work through this notebook.** Run the cells top to bottom. Each
exercise has a cell for you to fill in, then a cell of tests. Run the tests
first and watch them fail - read *what* they check, because the tests are the
specification. Then write code until they go green.

After you edit an exercise cell, **re-run it** before re-running its tests - the
tests use whatever is currently defined in the notebook.

In [ ]:
import ipytest
import pytest

# clean="test_*": before each test cell, forget test functions from earlier cells
# - but not other names starting with "Test", like milestone 2's TestCase class.
ipytest.autoconfig(clean="test_*")
from typing import Any, TypedDict

## 1. The state types

`SubmissionResult` is the outcome of running one submission against a problem's
test cases. `DrillState` is the state for one practice session.

`total=False` means every key is optional. That is what lets a node return
`{"attempts": 2}` instead of rebuilding the whole dict - so keep it.

In [ ]:
class SubmissionResult(TypedDict, total=False):
    """The outcome of running one submission against a problem's test cases.

    TODO: give this the following keys, with these types:
        passed        bool   - did every case pass?
        total         int    - how many cases ran
        passed_count  int    - how many passed
        fatal_error   str | None - set when the code did not run at all
    """

    # TODO: replace this with the real keys.
    _placeholder: Any


class DrillState(TypedDict, total=False):
    """State for one practice session.

    TODO: give this the following keys:
        problem_id   str
        attempts     int
        max_attempts int
        hints        list[str]
        submission   str
        result       SubmissionResult
        feedback     str
    """

    # TODO: replace this with the real keys.
    _placeholder: Any

In [ ]:
%%ipytest -qq

def test_state_declares_the_expected_keys():
    keys = set(DrillState.__annotations__)
    assert {"problem_id", "attempts", "max_attempts", "hints", "submission",
            "result", "feedback"} <= keys
    assert "_placeholder" not in keys, "replace the placeholder with the real keys"


def test_result_declares_the_expected_keys():
    keys = set(SubmissionResult.__annotations__)
    assert {"passed", "total", "passed_count", "fatal_error"} <= keys
    assert "_placeholder" not in keys


def test_state_keys_are_optional():
    """total=False is what lets a node return one key instead of the whole dict."""
    assert DrillState.__total__ is False

## 2. `merge_update`

This is the heart of how a graph accumulates work. Requirements:

- Return a **new** dict. Never mutate `state` - a caller may still be holding it,
  and LangGraph relies on old states staying intact for checkpointing.
- Keys in `update` overwrite keys in `state`.
- Keys absent from `update` are carried through unchanged.
- An empty update returns an equal copy of the state.

In [ ]:
def merge_update(state: dict, update: dict) -> dict:
    """Merge a node's partial update into the state and return the new state."""
    # TODO: implement.
    raise NotImplementedError("Milestone 1: implement merge_update")

In [ ]:
%%ipytest -qq

def test_update_overwrites_matching_keys():
    assert merge_update({"attempts": 1}, {"attempts": 2})["attempts"] == 2


def test_untouched_keys_survive():
    merged = merge_update({"problem_id": "two_sum", "attempts": 1}, {"attempts": 2})
    assert merged["problem_id"] == "two_sum"


def test_new_keys_are_added():
    assert merge_update({"attempts": 1}, {"feedback": "nice"})["feedback"] == "nice"


def test_empty_update_changes_nothing():
    state = {"attempts": 1, "hints": []}
    assert merge_update(state, {}) == state


def test_the_original_state_is_not_mutated():
    """LangGraph keeps old states for checkpointing. Mutating one corrupts history."""
    state = {"attempts": 1}
    merge_update(state, {"attempts": 99})
    assert state["attempts"] == 1


def test_merge_returns_a_new_object():
    state = {"attempts": 1}
    assert merge_update(state, {}) is not state

## 3. `bump_attempts`

Return the partial update that records one more attempt: exactly
`{"attempts": <one more than the current count>}`, treating a missing `attempts`
key as 0.

Note it returns *only* the changed key - not the whole state. That is the
convention every node in this project follows.

In [ ]:
def bump_attempts(state: dict) -> dict:
    """Return the partial update that records one more attempt."""
    # TODO: implement.
    raise NotImplementedError("Milestone 1: implement bump_attempts")

In [ ]:
%%ipytest -qq

def test_bump_returns_only_the_changed_key():
    assert bump_attempts({"attempts": 1, "problem_id": "two_sum"}) == {"attempts": 2}


def test_bump_treats_a_missing_count_as_zero():
    assert bump_attempts({}) == {"attempts": 1}

## Try it

Once the tests pass, run a node's worth of updates by hand. This is what LangGraph
does between every pair of nodes.

In [ ]:
state = {"problem_id": "two_sum", "attempts": 0}
for _ in range(3):
    state = merge_update(state, bump_attempts(state))
state